# 02b Narrative Sentiment

Precompute VADER sentiment on `cleaned_complaints.parquet` so the supervised notebooks can reuse a saved feature instead of recalculating it every run.


## Overview

This notebook reads the cleaned complaints dataset, computes a package-based sentiment score using VADER, and writes parquet outputs that can be reused by the final supervised notebooks.

Outputs:
- `data/processed/cleaned_complaints_vader.parquet`
- `data/processed/consumer_banking_relief_vader.parquet`


In [2]:
from pathlib import Path

import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
SOURCE_PATH = DATA_DIR / "cleaned_complaints.parquet"
FULL_OUTPUT_PATH = DATA_DIR / "cleaned_complaints_vader.parquet"
RELIEF_OUTPUT_PATH = DATA_DIR / "consumer_banking_relief_vader.parquet"

assert SOURCE_PATH.exists(), f"Expected cleaned parquet at {SOURCE_PATH}"


In [3]:
df = pd.read_parquet(SOURCE_PATH)
print(f"Loaded {len(df):,} rows and {df.shape[1]} columns from {SOURCE_PATH.name}")
df.head()


Loaded 1,133,355 rows and 19 columns from cleaned_complaints.parquet


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,cleaned_consumer_narrative
0,2019-12-26,Credit card or prepaid card,General-purpose credit card or charge card,"Advertising and marketing, including promotion...",Confusing or misleading advertising about the ...,[No Narrative],NaN,CAPITAL ONE FINANCIAL CORPORATION,CA,94025,NaN,Consent not provided,Web,2019-12-26,Closed with explanation,True,N/A,3477549,[No Narrative]
1,2019-12-20,Checking or savings account,Other banking product or service,Managing an account,Funds not handled or disbursed as instructed,[No Narrative],Company has responded to the consumer and the ...,WELLS FARGO & COMPANY,FL,33064,NaN,N/A,Referral,2019-12-23,Closed with explanation,True,N/A,3475858,[No Narrative]
2,2019-11-18,Credit card or prepaid card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,XXXX claimed they delivered a package to my ad...,NaN,DISCOVER BANK,MA,021XX,NaN,Consent provided,Web,2019-11-18,Closed with explanation,True,N/A,3442136,REDACTED claimed they delivered a package to m...
3,2020-06-05,Checking or savings account,Checking account,Managing an account,Problem using a debit or ATM card,[No Narrative],Company has responded to the consumer and the ...,"CITIBANK, N.A.",NY,10466,NaN,Consent not provided,Web,2020-06-05,Closed with explanation,True,N/A,3684669,[No Narrative]
4,2024-01-16,Credit card,General-purpose credit card or charge card,"Other features, terms, or problems",Add-on products and services,[No Narrative],Company has responded to the consumer and the ...,WELLS FARGO & COMPANY,TX,76179,NaN,Consent not provided,Web,2024-01-16,Closed with monetary relief,True,N/A,8161600,[No Narrative]


## Sentiment Source

We use `cleaned_consumer_narrative` when it exists because it is the cleaned text field. If it is missing, we fall back to `Consumer complaint narrative`. Narratives represented as `[No Narrative]` receive a neutral score of `0.0`.


In [4]:
analyzer = SentimentIntensityAnalyzer()

sentiment_text_col = "cleaned_consumer_narrative" if "cleaned_consumer_narrative" in df.columns else "Consumer complaint narrative"
text_series = (
    df[sentiment_text_col]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

no_narrative_mask = text_series.eq("") | text_series.eq("[No Narrative]")
df["narrative_sentiment_score"] = text_series.map(
    lambda value: analyzer.polarity_scores(value)["compound"] if value and value != "[No Narrative]" else 0.0
).astype(float)

display(
    pd.DataFrame(
        {
            "sentiment_source_column": [sentiment_text_col],
            "rows": [len(df)],
            "no_narrative_rows": [int(no_narrative_mask.sum())],
            "sentiment_min": [df["narrative_sentiment_score"].min()],
            "sentiment_mean": [df["narrative_sentiment_score"].mean()],
            "sentiment_max": [df["narrative_sentiment_score"].max()],
        }
    )
)


,sentiment_source_column,rows,no_narrative_rows,sentiment_min,sentiment_mean,sentiment_max
0,cleaned_consumer_narrative,1133355,592147,-1.0,-0.045677,1.0


## Write Outputs

The full enriched dataset is saved first. Then we also write the relief-focused subset used by the final supervised notebooks so those notebooks can load a precomputed sentiment feature directly.


In [5]:
relief_values = {
    "Closed with monetary relief",
    "Closed with non-monetary relief",
    "Closed with explanation",
}

relief_df = df.loc[df["Company response to consumer"].isin(relief_values)].copy()

df.to_parquet(FULL_OUTPUT_PATH, index=False)
relief_df.to_parquet(RELIEF_OUTPUT_PATH, index=False)

print(f"Saved full sentiment-enriched data to {FULL_OUTPUT_PATH}")
print(f"Saved relief subset with sentiment to {RELIEF_OUTPUT_PATH}")
print(f"Relief subset rows: {len(relief_df):,}")


Saved full sentiment-enriched data to C:\Users\ntamm\OneDrive\Documents\1_MADS\1_Classes\696 - Milestone 2\Project\cfpb_mortgage_complaints\data\processed\cleaned_complaints_vader.parquet
Saved relief subset with sentiment to C:\Users\ntamm\OneDrive\Documents\1_MADS\1_Classes\696 - Milestone 2\Project\cfpb_mortgage_complaints\data\processed\consumer_banking_relief_vader.parquet
Relief subset rows: 1,121,485
